# 06 · Exportar para o Tableau
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/segmentacao-clientes-olist/blob/main/notebooks/06_exportar_tableau.ipynb)

Gera três CSVs para montar o painel no **Tableau Public**:
1. `tableau_clientes.csv`: um cliente por linha, com segmento, R, F, V, estado, categoria preferida e nota média.
2. `tableau_mensal.csv`: pedidos e receita por mês e por segmento.
3. `tableau_segmentos.csv`: resumo por segmento, com a ação de CRM sugerida.

Os arquivos ficam na pasta `dados` do projeto no seu Google Drive.

In [1]:
import sqlite3, pandas as pd

try:                                    # no Colab: lê o banco do seu Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DADOS = '/content/drive/MyDrive/03_Portfolio/Projetos_ML/01_segmentacao-clientes-olist/dados'
except ModuleNotFoundError:             # fora do Colab: pasta dados/ do projeto
    DADOS = '../dados'

con = sqlite3.connect(f'{DADOS}/olist.db')

def q(sql):
    """Roda uma consulta SQL no banco da Olist e devolve um DataFrame."""
    return pd.read_sql(sql, con)


# ------------------------------------------------------------------------------
# Garante as tabelas rfm e segmentos (criadas nos notebooks 01 e 03).
# O Google Drive pode demorar a sincronizar o banco entre sessões do Colab; se elas
# não aparecerem aqui, são recriadas com exatamente o mesmo método do notebook 03.
# ------------------------------------------------------------------------------
tabelas = q("SELECT name FROM sqlite_master WHERE type='table'")['name'].tolist()
print('Banco em uso:', DADOS)
assert 'pedidos' in tabelas, f'O banco em {DADOS} está vazio: rode o notebook 00 antes.'

if 'segmentos' not in tabelas:
    print('Tabela segmentos não encontrada: recriando com o método do notebook 03...')
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    con.execute("""
    CREATE TABLE IF NOT EXISTS rfm AS
    WITH pago AS (SELECT order_id, SUM(payment_value) AS valor_pedido FROM pagamentos GROUP BY order_id),
    base AS (
        SELECT c.customer_unique_id, p.order_id, date(p.order_purchase_timestamp) AS data_compra, pg.valor_pedido
        FROM clientes c
        JOIN pedidos p ON c.customer_id = p.customer_id
        JOIN pago pg   ON p.order_id   = pg.order_id
        WHERE p.order_status = 'delivered'
    ),
    referencia AS (SELECT MAX(data_compra) AS ultima_data_base FROM base)
    SELECT b.customer_unique_id,
           CAST(julianday(r.ultima_data_base) - julianday(MAX(b.data_compra)) AS INT) AS recencia,
           COUNT(DISTINCT b.order_id) AS frequencia, ROUND(SUM(b.valor_pedido), 2) AS valor
    FROM base b CROSS JOIN referencia r
    GROUP BY b.customer_unique_id
    """)
    _rfm = q('SELECT * FROM rfm')
    _X = StandardScaler().fit_transform(pd.DataFrame({
        'recencia': _rfm['recencia'],
        'frequencia_log': np.log1p(_rfm['frequencia']),
        'valor_log': np.log1p(_rfm['valor'])}))
    _rfm['cluster'] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(_X)
    _p = _rfm.groupby('cluster')[['recencia', 'frequencia', 'valor']].mean()
    _nomes = {_p['frequencia'].idxmax(): 'Recorrentes'}
    _resto = _p.drop(list(_nomes))
    _nomes[_resto['recencia'].idxmax()] = 'Inativos'
    _resto = _resto.drop(list(_nomes), errors='ignore')
    _nomes[_resto['valor'].idxmax()] = 'Alto valor'
    _nomes[_resto['valor'].idxmin()] = 'Novos de baixo valor'
    _rfm['segmento'] = _rfm['cluster'].map(_nomes)
    _rfm[['customer_unique_id', 'recencia', 'frequencia', 'valor', 'segmento']].to_sql(
        'segmentos', con, if_exists='replace', index=False)
    con.commit()

print(q('SELECT segmento, COUNT(*) AS clientes FROM segmentos GROUP BY segmento ORDER BY segmento').to_string(index=False))

Banco em uso: ../dados
            segmento  clientes
          Alto valor     27832
            Inativos     27047
Novos de baixo valor     35677
         Recorrentes      2801


## 1. Base por cliente

In [2]:
clientes = q('''
WITH base AS (                                     -- pedidos entregues com a PESSOA
    SELECT c.customer_unique_id, c.customer_state, p.order_id,
           date(p.order_purchase_timestamp) AS data_compra
    FROM clientes c
    JOIN pedidos p ON c.customer_id = p.customer_id
    WHERE p.order_status = 'delivered'
),
estado AS (                                        -- estado da compra mais recente
    SELECT customer_unique_id, customer_state AS estado
    FROM (SELECT customer_unique_id, customer_state,
                 ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY data_compra DESC) AS ordem
          FROM base)
    WHERE ordem = 1
),
categoria AS (                                     -- categoria em que o cliente mais gastou
    SELECT customer_unique_id, categoria
    FROM (SELECT b.customer_unique_id,
                 COALESCE(ce.product_category_name_english, pr.product_category_name, 'sem_categoria') AS categoria,
                 ROW_NUMBER() OVER (PARTITION BY b.customer_unique_id ORDER BY SUM(i.price) DESC) AS ordem
          FROM base b
          JOIN itens_pedido i        ON b.order_id = i.order_id
          JOIN produtos pr           ON i.product_id = pr.product_id
          LEFT JOIN categorias_en ce ON pr.product_category_name = ce.product_category_name
          GROUP BY b.customer_unique_id, categoria)
    WHERE ordem = 1
),
datas AS (
    SELECT customer_unique_id, MIN(data_compra) AS primeira_compra, MAX(data_compra) AS ultima_compra
    FROM base GROUP BY customer_unique_id
),
nota AS (
    SELECT b.customer_unique_id, ROUND(AVG(a.review_score), 2) AS nota_media
    FROM base b JOIN avaliacoes a ON b.order_id = a.order_id
    GROUP BY b.customer_unique_id
)
SELECT s.customer_unique_id, s.segmento, s.recencia, s.frequencia, s.valor,
       e.estado, ca.categoria AS categoria_preferida,
       d.primeira_compra, d.ultima_compra, n.nota_media
FROM segmentos s
LEFT JOIN estado e     ON s.customer_unique_id = e.customer_unique_id
LEFT JOIN categoria ca ON s.customer_unique_id = ca.customer_unique_id
LEFT JOIN datas d      ON s.customer_unique_id = d.customer_unique_id
LEFT JOIN nota n       ON s.customer_unique_id = n.customer_unique_id
''')

print(clientes.shape)
print('% faltantes:'); print((clientes.isna().mean() * 100).round(2))
clientes.head()

(93357, 10)
% faltantes:
customer_unique_id     0.00
segmento               0.00
recencia               0.00
frequencia             0.00
valor                  0.00
estado                 0.00
categoria_preferida    0.00
primeira_compra        0.00
ultima_compra          0.00
nota_media             0.65
dtype: float64


,customer_unique_id,segmento,recencia,frequencia,valor,estado,categoria_preferida,primeira_compra,ultima_compra,nota_media
0,0000366f3b9a7992bf8c76cfdf3221e2,Alto valor,111,1,141.90,SP,bed_bath_table,2018-05-10,2018-05-10,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,Novos de baixo valor,114,1,27.19,SP,health_beauty,2018-05-07,2018-05-07,4.0
2,0000f46a3911fa3c0805444483337064,Inativos,537,1,86.22,SC,stationery,2017-03-10,2017-03-10,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,Inativos,321,1,43.62,PA,telephony,2017-10-12,2017-10-12,4.0
4,0004aac84e0df4da2b147fca70cf8255,Alto valor,288,1,196.89,SP,telephony,2017-11-14,2017-11-14,5.0


## 2. Série mensal por segmento

In [3]:
mensal = q('''
WITH pago AS (SELECT order_id, SUM(payment_value) AS valor FROM pagamentos GROUP BY order_id)
SELECT strftime('%Y-%m-01', p.order_purchase_timestamp) AS mes,
       s.segmento,
       COUNT(DISTINCT p.order_id)     AS pedidos,
       ROUND(SUM(pg.valor), 2)        AS receita
FROM segmentos s
JOIN clientes c ON s.customer_unique_id = c.customer_unique_id
JOIN pedidos p  ON c.customer_id = p.customer_id
JOIN pago pg    ON p.order_id = pg.order_id
WHERE p.order_status = 'delivered'
GROUP BY mes, s.segmento
ORDER BY mes, s.segmento
''')
mensal.tail(8)

,mes,segmento,pedidos,receita
60,2018-06-01,Novos de baixo valor,3417,250507.13
61,2018-06-01,Recorrentes,291,43084.11
62,2018-07-01,Alto valor,2364,726335.77
63,2018-07-01,Novos de baixo valor,3497,255055.26
64,2018-07-01,Recorrentes,298,46512.83
65,2018-08-01,Alto valor,2226,665285.20
66,2018-08-01,Novos de baixo valor,3884,283629.64
67,2018-08-01,Recorrentes,241,36499.44


## 2b. Nome do estado e resumo por segmento
O Tableau reconhece melhor o mapa com o nome completo do estado e o país. O resumo por segmento alimenta os cartões e a tabela de ações do painel.

In [4]:
UF = {'AC':'Acre','AL':'Alagoas','AP':'Amapá','AM':'Amazonas','BA':'Bahia','CE':'Ceará','DF':'Distrito Federal',
      'ES':'Espírito Santo','GO':'Goiás','MA':'Maranhão','MT':'Mato Grosso','MS':'Mato Grosso do Sul',
      'MG':'Minas Gerais','PA':'Pará','PB':'Paraíba','PR':'Paraná','PE':'Pernambuco','PI':'Piauí',
      'RJ':'Rio de Janeiro','RN':'Rio Grande do Norte','RS':'Rio Grande do Sul','RO':'Rondônia','RR':'Roraima',
      'SC':'Santa Catarina','SP':'São Paulo','SE':'Sergipe','TO':'Tocantins'}
clientes['estado_nome'] = clientes['estado'].map(UF)
clientes['pais'] = 'Brazil'
print('Estados sem nome:', clientes['estado_nome'].isna().sum())

ACOES = {
    'Recorrentes':          'Programa de fidelidade, acesso antecipado a promoções, recomendações por histórico',
    'Alto valor':           'Prioridade nº 1 para a 2ª compra: cupom na categoria comprada, frete grátis acima de um valor mínimo',
    'Novos de baixo valor': 'Aumentar o ticket: kits, frete grátis por valor mínimo, produtos complementares',
    'Inativos':             'Reativação de baixo custo (e-mail, push); testar antes de investir',
}
resumo = (clientes.groupby('segmento')
          .agg(clientes=('customer_unique_id', 'count'), receita=('valor', 'sum'),
               recencia_media=('recencia', 'mean'), frequencia_media=('frequencia', 'mean'),
               valor_medio=('valor', 'mean'), nota_media=('nota_media', 'mean'))
          .reset_index())
resumo['pct_clientes'] = 100 * resumo['clientes'] / resumo['clientes'].sum()
resumo['pct_receita']  = 100 * resumo['receita'] / resumo['receita'].sum()
resumo['acao_crm'] = resumo['segmento'].map(ACOES)
resumo = resumo.round(2).sort_values('receita', ascending=False)
resumo

Estados sem nome: 0


,segmento,clientes,receita,recencia_media,frequencia_media,valor_medio,nota_media,pct_clientes,pct_receita,acao_crm
0,Alto valor,27832,8860484.49,172.90,1.00,318.36,4.04,29.81,57.45,Prioridade nº 1 para a 2ª compra: cupom na cat...
1,Inativos,27047,3234099.22,424.73,1.00,119.57,4.22,28.97,20.97,"Reativação de baixo custo (e-mail, push); test..."
2,Novos de baixo valor,35677,2463520.85,147.28,1.00,69.05,4.19,38.22,15.97,"Aumentar o ticket: kits, frete grátis por valo..."
3,Recorrentes,2801,864357.21,219.81,2.11,308.59,4.21,3.00,5.60,"Programa de fidelidade, acesso antecipado a pr..."


## 3. Conferir e salvar

In [5]:
# Conferências: um cliente por linha e mesma receita nas duas bases
assert clientes['customer_unique_id'].is_unique
print('Clientes:', len(clientes), '| receita (clientes):', round(clientes['valor'].sum(), 2),
      '| receita (mensal):', round(mensal['receita'].sum(), 2))

clientes.to_csv(f'{DADOS}/tableau_clientes.csv', index=False)
mensal.to_csv(f'{DADOS}/tableau_mensal.csv', index=False)
resumo.to_csv(f'{DADOS}/tableau_segmentos.csv', index=False)
print('Arquivos salvos em', DADOS)

Clientes: 93357 | receita (clientes): 15422461.77 | receita (mensal): 15422461.77
Arquivos salvos em ../dados


## Próximo passo: o painel no Tableau Public
1. Baixe os três CSVs da pasta `dados` no Google Drive.
2. No Tableau Public, conecte `tableau_clientes.csv` (texto) e adicione `tableau_mensal.csv` e `tableau_segmentos.csv` como outras fontes.
3. **Ideias de painel:** cartões com clientes e receita por segmento · mapa por estado · evolução mensal por segmento · categoria preferida por segmento · tabela de ações de CRM.

## Gravar no Google Drive
Fecha o banco e força o Drive a gravar as mudanças agora. Sem isso, o próximo notebook, que roda em outra sessão do Colab, pode não enxergar as tabelas novas.

In [6]:
con.commit()
con.close()
try:
    drive.flush_and_unmount()      # só existe no Colab
    print('Banco gravado no Google Drive.')
except NameError:
    print('Fora do Colab: nada a sincronizar.')

Fora do Colab: nada a sincronizar.
